In [1]:
"""
Advanced transport-cost model for reseller inbound LPG cost (filling -> reseller, truck).

This cell replaces the previous simplified rule based only on travel time/1000.
It computes:
    cost_res_kg_in = cost_fil_kg_out_reference + transport_cost_kg

where transport_cost_kg includes:
- operating component (distance-based variable + labor),
- fixed annual truck cost share allocated by demand within each filling point catchment.

Data sources:
- chain_with_cost.gpkg (resell, filling)
- sell_point_clients.gpkg (reseller clients)
- filling_point_clients.gpkg (total_fil_clients for each filling point)

Notes:
- values with invalid reference/time/demand inputs are kept as NaN
- time >= 7000 min is treated as invalid/unassigned
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

# =============================================================================
# PATHS AND LAYERS
# =============================================================================
DATA_DIR = Path("dataset_big")
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
SELL_POINT_GPKG = DATA_DIR / "sell_point_clients.gpkg"
FILLING_POINT_GPKG = DATA_DIR / "filling_point_clients.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
SELL_POINT_LAYER = "sell_point_clients"
FILLING_POINT_LAYER = "filling_point_clients"

# =============================================================================
# KEYS AND COLUMNS
# =============================================================================
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_OUT_COL = "cost_fil_kg_out"
RESELL_COST_IN_COL = "cost_res_kg_in"
FILL_ORIGIN_COST_OUT_COL = "cost_fil_kg_out_reference"
SELL_CLIENTS_COL = "clients"
FILL_TOTAL_CLIENTS_COL = "total_fil_clients"

MAX_VALID_TIME_MIN = 7000.0

# =============================================================================
# ONSTOVE DEMAND DEFAULTS (kept explicit for visibility)
# =============================================================================
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY = 0.5
ENERGY_CONTENT_MJ_PER_KG = 45.5

# =============================================================================
# TRANSPORT MODEL PARAMETERS
# =============================================================================
# ---- From paper ----
truck_capacity_kg = 9996
utilization_factor = 0.85
truck_speed_kmh = 30
variable_cost_per_km = 0.540
driver_annual_salary_usd = 2604
salary_multiplier = 1.18
hours_per_day = 8
discount_rate = 0.10
truck_overnight_cost_usd = 170200
truck_life_years = 15
license_cost_5y_usd = 418

# ---- Assumptions ----
days_per_year = 330
fixed_loading_unloading_hours = 1.0

# =============================================================================
# HELPERS
# =============================================================================
def annual_lpg_demand_kg(
    num_clients: pd.Series | np.ndarray | float,
    meals_per_day: float = MEALS_PER_DAY,
    energy_per_meal: float = ENERGY_PER_MEAL_MJ,
    efficiency: float = EFFICIENCY,
    energy_content: float = ENERGY_CONTENT_MJ_PER_KG,
) -> pd.Series | np.ndarray | float:
    annual_energy_mj = num_clients * meals_per_day * 365.0 * energy_per_meal
    annual_mass_kg = annual_energy_mj / (efficiency * energy_content)
    return annual_mass_kg


def crf(rate: float, years: int) -> float:
    return (rate * (1.0 + rate) ** years) / ((1.0 + rate) ** years - 1.0)


effective_load_kg = truck_capacity_kg * utilization_factor
driver_hourly_cost_usd = (
    driver_annual_salary_usd * salary_multiplier / (hours_per_day * days_per_year)
)
crf_truck = crf(discount_rate, truck_life_years)
crf_license = crf(discount_rate, 5)
annual_capital_cost_usd = truck_overnight_cost_usd * crf_truck
annual_license_cost_usd = license_cost_5y_usd * crf_license
fixed_annual_truck_cost_usd = annual_capital_cost_usd + annual_license_cost_usd


def operating_cost_per_kg(travel_time_minutes: pd.Series) -> pd.Series:
    round_trip_hours = (travel_time_minutes * 2.0 / 60.0) + fixed_loading_unloading_hours
    round_trip_distance_km = (travel_time_minutes * 2.0 / 60.0) * truck_speed_kmh
    variable_cost = variable_cost_per_km * round_trip_distance_km
    labour_cost = driver_hourly_cost_usd * round_trip_hours
    trip_operating_cost_usd = variable_cost + labour_cost
    return trip_operating_cost_usd / effective_load_kg


print("[1/5] Loading required layers...")
resell = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(CHAIN_GPKG, layer=FILLING_LAYER)
sell_points = gpd.read_file(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
filling_points = gpd.read_file(FILLING_POINT_GPKG, layer=FILLING_POINT_LAYER)

if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")
if filling.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty.")
if sell_points.empty:
    raise RuntimeError(f"Layer '{SELL_POINT_LAYER}' is empty.")
if filling_points.empty:
    raise RuntimeError(f"Layer '{FILLING_POINT_LAYER}' is empty.")

for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'.")
for c in [ID_COL, FILL_COST_OUT_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'.")
for c in [ID_COL, SELL_CLIENTS_COL]:
    if c not in sell_points.columns:
        raise KeyError(f"Missing column '{c}' in layer '{SELL_POINT_LAYER}'.")
for c in [ID_COL, FILL_TOTAL_CLIENTS_COL]:
    if c not in filling_points.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_POINT_LAYER}'.")

print("[2/5] Building reference maps (costs and demand)...")
fid = pd.to_numeric(filling[ID_COL], errors="coerce")
fcost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
map_filling_cost_out = dict(zip(fid.dropna().astype(np.int64), fcost_out.astype(float)))

sell_id = pd.to_numeric(sell_points[ID_COL], errors="coerce")
sell_clients = pd.to_numeric(sell_points[SELL_CLIENTS_COL], errors="coerce")
sell_clients = sell_clients.fillna(0.0).astype(float)
map_reseller_clients = dict(zip(sell_id.dropna().astype(np.int64), sell_clients.astype(float)))

fillp_id = pd.to_numeric(filling_points[ID_COL], errors="coerce")
fillp_total_clients = pd.to_numeric(filling_points[FILL_TOTAL_CLIENTS_COL], errors="coerce")
fillp_total_clients = fillp_total_clients.fillna(0.0).astype(float)
map_filling_total_clients = dict(zip(fillp_id.dropna().astype(np.int64), fillp_total_clients.astype(float)))

print("[3/5] Computing transport cost terms and inbound cost...")
rid = pd.to_numeric(resell[ID_COL], errors="coerce")
fref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")

fill_cost_out_ref = fref.map(map_filling_cost_out)
reseller_clients = rid.map(map_reseller_clients)
filling_total_clients = fref.map(map_filling_total_clients)

annual_demand_reseller_kg = pd.to_numeric(
    annual_lpg_demand_kg(reseller_clients), errors="coerce"
).astype(float)
annual_demand_filling_total_kg = pd.to_numeric(
    annual_lpg_demand_kg(filling_total_clients), errors="coerce"
).astype(float)

op_cost_kg = operating_cost_per_kg(tmin)
fixed_share_cost_kg = fixed_annual_truck_cost_usd / annual_demand_filling_total_kg
transport_cost_kg = op_cost_kg + fixed_share_cost_kg

valid = (
    fref.notna()
    & (fref > 0)
    & np.isfinite(tmin)
    & (tmin >= 0)
    & (tmin < MAX_VALID_TIME_MIN)
    & fill_cost_out_ref.notna()
    & np.isfinite(fill_cost_out_ref)
    & (fill_cost_out_ref >= 0)
    & annual_demand_reseller_kg.notna()
    & np.isfinite(annual_demand_reseller_kg)
    & (annual_demand_reseller_kg > 0)
    & annual_demand_filling_total_kg.notna()
    & np.isfinite(annual_demand_filling_total_kg)
    & (annual_demand_filling_total_kg > 0)
    & np.isfinite(transport_cost_kg)
    & (transport_cost_kg >= 0)
)

new_cost_in = np.where(
    valid,
    fill_cost_out_ref + transport_cost_kg,
    np.nan,
)

print("[4/5] Overwriting reseller inbound cost in chain_with_cost.gpkg...")
resell[FILL_ORIGIN_COST_OUT_COL] = pd.to_numeric(fill_cost_out_ref, errors="coerce").astype(float)
resell[RESELL_COST_IN_COL] = np.nan
resell.loc[valid, RESELL_COST_IN_COL] = pd.to_numeric(
    pd.Series(new_cost_in, index=resell.index)[valid], errors="coerce"
).astype(float)
resell[RESELL_COST_IN_COL] = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce").astype(float)
resell = gpd.GeoDataFrame(resell, geometry="geometry", crs=resell.crs)
resell.to_file(CHAIN_GPKG, layer=RESELL_LAYER, driver="GPKG")

print("[5/5] QA summary and NaN diagnostics...")
n_total = int(len(resell))
n_valid = int(valid.sum())
n_nan = int((~np.isfinite(resell[RESELL_COST_IN_COL])).sum())

nan_due_to_ref = int((~(fref.notna() & (fref > 0) & fill_cost_out_ref.notna() & np.isfinite(fill_cost_out_ref))).sum())
nan_due_to_time = int((~(np.isfinite(tmin) & (tmin >= 0) & (tmin < MAX_VALID_TIME_MIN))).sum())
nan_due_to_reseller_demand = int((~(annual_demand_reseller_kg.notna() & np.isfinite(annual_demand_reseller_kg) & (annual_demand_reseller_kg > 0))).sum())
nan_due_to_filling_total_demand = int((~(annual_demand_filling_total_kg.notna() & np.isfinite(annual_demand_filling_total_kg) & (annual_demand_filling_total_kg > 0))).sum())

print(f"Updated rows (valid): {n_valid:,}/{n_total:,}")
print(f"NaN rows in {RESELL_COST_IN_COL}: {n_nan:,}")
print("NaN diagnostics (overlapping causes, not mutually exclusive):")
print(f"- invalid/missing filling reference or filling cost source: {nan_due_to_ref:,}")
print(f"- invalid travel time (<0 or >= {MAX_VALID_TIME_MIN:.0f} or NaN): {nan_due_to_time:,}")
print(f"- invalid/missing reseller demand (from sell_point_clients.clients): {nan_due_to_reseller_demand:,}")
print(f"- invalid/missing filling total demand (from filling_point_clients.total_fil_clients): {nan_due_to_filling_total_demand:,}")

if n_valid > 0:
    s = pd.to_numeric(resell.loc[valid, RESELL_COST_IN_COL], errors="coerce").astype(float)
    print(
        "Sanity (valid rows) min/median/mean/p95/max: "
        f"{float(np.nanmin(s)):.4f} / "
        f"{float(np.nanmedian(s)):.4f} / "
        f"{float(np.nanmean(s)):.4f} / "
        f"{float(np.nanpercentile(s, 95)):.4f} / "
        f"{float(np.nanmax(s)):.4f}"
    )
else:
    raise RuntimeError("No valid reseller rows found for advanced inbound cost update.")

print(f"Output file: {CHAIN_GPKG}")
print(f"Column overwritten: {RESELL_COST_IN_COL}")
print(f"Kept helper column: {FILL_ORIGIN_COST_OUT_COL}")

[1/5] Loading required layers...
[2/5] Building reference maps (costs and demand)...
[3/5] Computing transport cost terms and inbound cost...
[4/5] Overwriting reseller inbound cost in chain_with_cost.gpkg...
[5/5] QA summary and NaN diagnostics...
Updated rows (valid): 1,909/2,413
NaN rows in cost_res_kg_in: 504
NaN diagnostics (overlapping causes, not mutually exclusive):
- invalid/missing filling reference or filling cost source: 0
- invalid travel time (<0 or >= 7000 or NaN): 0
- invalid/missing reseller demand (from sell_point_clients.clients): 504
- invalid/missing filling total demand (from filling_point_clients.total_fil_clients): 1
Sanity (valid rows) min/median/mean/p95/max: 0.6006 / 0.6052 / 0.6079 / 0.6212 / 0.8379
Output file: dataset_big\chain_with_cost.gpkg
Column overwritten: cost_res_kg_in
Kept helper column: cost_fil_kg_out_reference


In [2]:
"""
Compute reseller outbound cost from reseller inbound cost.

Formula (temporary rule):
    cost_res_kg_out = 1.10 * cost_res_kg_in

Notes:
- applied only where cost_res_kg_in is finite and non-negative
- invalid inbound values keep NaN in outbound
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
OUT_MARKUP_FACTOR = 1.10

print("[1/3] Loading reseller layer...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")
if RESELL_COST_IN_COL not in resell.columns:
    raise KeyError(f"Missing required column '{RESELL_COST_IN_COL}' in layer '{RESELL_LAYER}'.")

print("[2/3] Computing outbound reseller cost...")
cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce")
valid = np.isfinite(cost_in) & (cost_in >= 0)
cost_out = np.where(valid, cost_in * OUT_MARKUP_FACTOR, np.nan)

if int(valid.sum()) == 0:
    raise RuntimeError("No valid inbound reseller costs found to compute outbound costs.")

print("[3/3] Writing outbound reseller cost to GeoPackage...")
resell[RESELL_COST_OUT_COL] = pd.to_numeric(pd.Series(cost_out, index=resell.index), errors="coerce").astype(float)
resell = gpd.GeoDataFrame(resell, geometry="geometry", crs=resell.crs)
resell.to_file(WORK_GPKG, layer=RESELL_LAYER, driver="GPKG")

print("Done.")
print(f"Output file: {WORK_GPKG}")
print(f"Column updated: {RESELL_COST_OUT_COL}")
print(f"Markup factor applied: {OUT_MARKUP_FACTOR:.2f}")
print(
    "Sanity (valid rows) min/median/mean/p95/max: "
    f"{float(np.nanmin(resell.loc[valid, RESELL_COST_OUT_COL])):.4f} / "
    f"{float(np.nanmedian(resell.loc[valid, RESELL_COST_OUT_COL])):.4f} / "
    f"{float(np.nanmean(resell.loc[valid, RESELL_COST_OUT_COL])):.4f} / "
    f"{float(np.nanpercentile(resell.loc[valid, RESELL_COST_OUT_COL], 95)):.4f} / "
    f"{float(np.nanmax(resell.loc[valid, RESELL_COST_OUT_COL])):.4f}"
)

[1/3] Loading reseller layer...
[2/3] Computing outbound reseller cost...
[3/3] Writing outbound reseller cost to GeoPackage...
Done.
Output file: dataset_big\chain_with_cost.gpkg
Column updated: cost_res_kg_out
Markup factor applied: 1.10
Sanity (valid rows) min/median/mean/p95/max: 0.6607 / 0.6658 / 0.6687 / 0.6833 / 0.9217


In [3]:
"""
Final QA stats for 4.4 outputs with split in/out costs.

Checks:
1) filling points have valid cost_fil_kg_out and cost_fil_kg_in
2) resell points have valid filling_reference and travel stats
3) resell points have valid cost_res_kg_in and cost_res_kg_out
4) outbound/inbound reseller ratio is coherent with configured markup
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
LEGACY_COST_COL = "cost_kg"
OUT_MARKUP_FACTOR = 1.10

print("[1/4] Loading layers...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")

for c in [ID_COL, FILL_COST_OUT_COL, FILL_COST_IN_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'")
for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL, RESELL_COST_IN_COL, RESELL_COST_OUT_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'")

fill_id = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_cost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
fill_cost_in = pd.to_numeric(filling[FILL_COST_IN_COL], errors="coerce")
res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce")
res_cost_out = pd.to_numeric(resell[RESELL_COST_OUT_COL], errors="coerce")

print("[2/4] QA on filling in/out costs...")
n_fill = len(filling)
fill_out_valid = np.isfinite(fill_cost_out)
fill_in_valid = np.isfinite(fill_cost_in)
fill_out_nonneg = fill_out_valid & (fill_cost_out >= 0)
fill_in_nonneg = fill_in_valid & (fill_cost_in >= 0)

print("=== FILLING COST QA ===")
print(f"Filling points total                  : {n_fill:,}")
print(f"Finite {FILL_COST_OUT_COL}            : {int(fill_out_valid.sum()):,}")
print(f"Finite {FILL_COST_IN_COL}             : {int(fill_in_valid.sum()):,}")
print(f"{FILL_COST_OUT_COL} < 0               : {int((~fill_out_nonneg).sum()):,}")
print(f"{FILL_COST_IN_COL} < 0                : {int((~fill_in_nonneg).sum()):,}")
if fill_out_valid.any():
    print(
        f"{FILL_COST_OUT_COL} min/median/mean/max: "
        f"{float(np.nanmin(fill_cost_out)):.4f} / {float(np.nanmedian(fill_cost_out)):.4f} / "
        f"{float(np.nanmean(fill_cost_out)):.4f} / {float(np.nanmax(fill_cost_out)):.4f}"
    )
if fill_in_valid.any():
    print(
        f"{FILL_COST_IN_COL} min/median/mean/max : "
        f"{float(np.nanmin(fill_cost_in)):.4f} / {float(np.nanmedian(fill_cost_in)):.4f} / "
        f"{float(np.nanmean(fill_cost_in)):.4f} / {float(np.nanmax(fill_cost_in)):.4f}"
    )
if LEGACY_COST_COL in filling.columns:
    legacy_fill = pd.to_numeric(filling[LEGACY_COST_COL], errors="coerce")
    print(f"Legacy {LEGACY_COST_COL} finite count : {int(np.isfinite(legacy_fill).sum()):,}/{n_fill:,}")

print("\n[3/4] QA on reseller filling reference, travel stats, and costs...")
n_res = len(resell)
fill_id_set = set(fill_id[np.isfinite(fill_id)].astype(np.int64).tolist())
ref_finite = np.isfinite(res_ref)
ref_positive = ref_finite & (res_ref > 0)
ref_exists = ref_positive & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()

time_valid = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
ref_and_time_valid = ref_exists & time_valid

res_in_valid = np.isfinite(res_cost_in) & (res_cost_in >= 0)
res_out_valid = np.isfinite(res_cost_out) & (res_cost_out >= 0)
res_in_linked = ref_and_time_valid & res_in_valid
res_out_linked = ref_and_time_valid & res_out_valid

print("=== RESELLER REFERENCE QA ===")
print(f"Resell points total                    : {n_res:,}")
print(f"Resell with finite filling_reference   : {int(ref_finite.sum()):,}")
print(f"Resell with existing filling_reference : {int(ref_exists.sum()):,}")
print(f"Resell with valid reference + time     : {int(ref_and_time_valid.sum()):,}")
print(f"Missing/invalid reference              : {int((~ref_exists).sum()):,}")
if ref_and_time_valid.any():
    t = res_tmin[ref_and_time_valid].to_numpy(dtype=float)
    print(
        "Travel time min/median/mean/p95/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / {float(np.nanmean(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanmax(t)):.2f}"
    )

print("=== RESELLER COST QA ===")
print(f"Finite {RESELL_COST_IN_COL}                    : {int(np.isfinite(res_cost_in).sum()):,}/{n_res:,}")
print(f"Finite {RESELL_COST_OUT_COL}                   : {int(np.isfinite(res_cost_out).sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} valid + linked            : {int(res_in_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_OUT_COL} valid + linked           : {int(res_out_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} NaN rows                  : {int((~np.isfinite(res_cost_in)).sum()):,}")
print(f"{RESELL_COST_OUT_COL} NaN rows                 : {int((~np.isfinite(res_cost_out)).sum()):,}")
if res_in_valid.any():
    print(
        f"{RESELL_COST_IN_COL} min/median/mean/p95/max      : "
        f"{float(np.nanmin(res_cost_in)):.4f} / {float(np.nanmedian(res_cost_in)):.4f} / "
        f"{float(np.nanmean(res_cost_in)):.4f} / {float(np.nanpercentile(res_cost_in, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_in)):.4f}"
    )
if res_out_valid.any():
    print(
        f"{RESELL_COST_OUT_COL} min/median/mean/p95/max     : "
        f"{float(np.nanmin(res_cost_out)):.4f} / {float(np.nanmedian(res_cost_out)):.4f} / "
        f"{float(np.nanmean(res_cost_out)):.4f} / {float(np.nanpercentile(res_cost_out, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_out)):.4f}"
    )

ratio_valid = res_in_valid & res_out_valid & (res_cost_in > 0)
if ratio_valid.any():
    ratio = (res_cost_out[ratio_valid] / res_cost_in[ratio_valid]).to_numpy(dtype=float)
    print("=== MARKUP CONSISTENCY QA ===")
    print(
        "Observed ratio min/median/mean/max: "
        f"{float(np.nanmin(ratio)):.6f} / {float(np.nanmedian(ratio)):.6f} / "
        f"{float(np.nanmean(ratio)):.6f} / {float(np.nanmax(ratio)):.6f}"
    )
    print(f"Target markup ratio: {OUT_MARKUP_FACTOR:.6f}")

if (resell.crs is not None) and (filling.crs is not None) and (str(resell.crs) == str(filling.crs)):
    fill_geom = filling[[ID_COL, "geometry"]].copy()
    fill_geom[ID_COL] = pd.to_numeric(fill_geom[ID_COL], errors="coerce")
    merged = resell[[ID_COL, FILL_REF_COL, "geometry"]].copy()
    merged[ID_COL] = pd.to_numeric(merged[ID_COL], errors="coerce")
    merged[FILL_REF_COL] = pd.to_numeric(merged[FILL_REF_COL], errors="coerce")
    merged = merged.merge(fill_geom, left_on=FILL_REF_COL, right_on=ID_COL, how="left", suffixes=("_res", "_fill"))
    good_geom = merged["geometry_res"].notna() & merged["geometry_fill"].notna()
    if good_geom.any():
        dist_km = merged.loc[good_geom, "geometry_res"].distance(merged.loc[good_geom, "geometry_fill"]).astype(float) / 1000.0
        print(
            "Straight-line distance min/median/mean/p95/max (km): "
            f"{float(np.nanmin(dist_km)):.2f} / {float(np.nanmedian(dist_km)):.2f} / {float(np.nanmean(dist_km)):.2f} / "
            f"{float(np.nanpercentile(dist_km, 95)):.2f} / {float(np.nanmax(dist_km)):.2f}"
        )

print("\n[4/4] QA completed.")

[1/4] Loading layers...
[2/4] QA on filling in/out costs...
=== FILLING COST QA ===
Filling points total                  : 375
Finite cost_fil_kg_out            : 375
Finite cost_fil_kg_in             : 375
cost_fil_kg_out < 0               : 0
cost_fil_kg_in < 0                : 0
cost_fil_kg_out min/median/mean/max: 0.6000 / 0.6000 / 0.6000 / 0.6000
cost_fil_kg_in min/median/mean/max : 0.5000 / 0.5000 / 0.5000 / 0.5000
Legacy cost_kg finite count : 375/375

[3/4] QA on reseller filling reference, travel stats, and costs...
=== RESELLER REFERENCE QA ===
Resell points total                    : 2,413
Resell with finite filling_reference   : 2,413
Resell with existing filling_reference : 2,413
Resell with valid reference + time     : 2,413
Missing/invalid reference              : 0
Travel time min/median/mean/p95/max (min): 0.00 / 4.00 / 19.28 / 81.84 / 1087.94
=== RESELLER COST QA ===
Finite cost_res_kg_in                    : 1,909/2,413
Finite cost_res_kg_out                   : 1,9